# EcoSort AI - Simplified Local Demo

This notebook explains the final application in four modules. The original EcoSort-Local folder remains the full development archive. This is a structural simplification, not a newly trained model or a new accuracy claim.

## 1. Check the local environment

Run this notebook from the EcoSort-Demo folder and select its .venv Python interpreter. PyTorch uses the Apple GPU (MPS) when available. Model weights are stored separately from source code.

In [ ]:
from pathlib import Path
import torch

assert Path('app.py').is_file(), 'Open this notebook from EcoSort-Demo.'
print('Apple GPU available:', torch.backends.mps.is_available())
print('Project:', Path.cwd())

## 2. Understand the four files

- app.py: receives an uploaded or captured photo, calls the classifier, and displays the result.
- classifier.py: runs the pinned, frozen SigLIP2 model against item descriptions and flags uncertain results.
- disposal_rules.py: converts a suggested or user-confirmed item into City of Sydney disposal guidance.
- evaluate.py: checks safety-related examples and compares the original 60-photo panel against its recorded decisions.

The pretrained model was not trained on our historical 147,303-image project dataset. Relative candidate scores are not calibrated probabilities.

In [ ]:
from app import EcoSort, verify_release

verify_release()
engine = EcoSort()
print('Verified source files and loaded the final frozen model.')

## 3. Identify one photo

Set photo_path to a local image. The application processes the full image, compares image and text representations, selects an item, and applies an uncertainty check. Busy backgrounds, transparent objects and soft plastic remain known weaknesses. Do not treat a confident suggestion as proof of correct disposal.

In [ ]:
from PIL import Image

photo_path = None  # Replace None with Path('/absolute/path/to/photo.jpg').
if photo_path is not None:
    with Image.open(photo_path) as image:
        scores, decision, guidance = engine.classify(image)
    print(decision)
    print(guidance)
    print(sorted(scores.items(), key=lambda item: item[1], reverse=True)[:5])

## 4. Separate recognition from disposal rules

Material alone does not determine the bin. A plastic bottle and a soft plastic bag can require different routes. User confirmation overrides the item interpretation, not the model's material scores. Official links are included in the returned guidance; food-scraps instructions depend on access to the local service.

In [ ]:
from disposal_rules import predicted_item_guidance

for item in ('plastic drink bottle', 'plastic bag'):
    route, guidance = predicted_item_guidance('plastic', item)
    print(item, '->', route)

## 5. Read the regression evidence

The 60 photos were already observed during development. This regression checks that simplification preserves their decisions, including mistakes; it is not an independent estimate of real-world accuracy. Historical category results were 19/20, 18/20 and 20/20; joint category-and-disposal results were 19/20, 18/20 and 18/20. Dataset validation/test results from older trained models must not be attributed to this frozen model.

To rerun, use this environment's Python with evaluate.py --images /path/to/original/photos, or evaluate.py --restore to retrieve the recorded public images. Source hashes must match.

In [ ]:
import json

record = json.loads(Path('evidence/refactor_verified.json').read_text())
print(record['kind'])
print('Matching original decisions:', record['parity_count'])
print(json.dumps(record['original_groups'], indent=2))

## 6. Launch the English interface

The interface supports file upload and webcam capture, relative candidate scores, manual review, user item confirmation and City of Sydney disposal guidance. It runs locally on port 7862; the original application on port 7861 is unchanged. Run either this cell or Start EcoSort.command, not two separate servers.

In [ ]:
from app import launch

demo = launch(prevent_thread_lock=True)